<a href="https://colab.research.google.com/github/leonardomerolla-droid/cyber-lab/blob/main/deepFake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!git clone https://github.com/AliaksandrSiarohin/first-order-model

Cloning into 'first-order-model'...
remote: Enumerating objects: 397, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 397 (delta 40), reused 35 (delta 35), pack-reused 343 (from 2)
Receiving objects: 100% (397/397), 72.19 MiB | 36.58 MiB/s, done.
Resolving deltas: 100% (206/206), done.


In [16]:
cd first-order-model

/content/first-order-model/first-order-model/first-order-model


In [17]:
from google.colab import drive
drive.mount('/content/gdrive/')

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [18]:
ls /content/gdrive/My\ Drive/DeepFake/

bob2.png  deepFake.ipynb  Obama.mp4  README.md  vox-adv-cpk.pth.tar


In [19]:
import imageio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from skimage.transform import resize
from IPython.display import HTML
import warnings
warnings.filterwarnings("ignore")

source_image = imageio.imread('/content/gdrive/My Drive/DeepFake/bob2.png')
driving_video = imageio.mimread('/content/gdrive/My Drive/DeepFake/Obama.mp4')
#Resize image and video to 256x256
source_image = resize(source_image, (256, 256))[..., :3]
driving_video = [resize(frame, (256, 256))[..., :3] for frame in driving_video]


In [20]:
from demo import load_checkpoints
generator, kp_detector = load_checkpoints(config_path='config/vox-256.yaml',
                            checkpoint_path='/content/gdrive/My Drive/DeepFake/vox-adv-cpk.pth.tar')

In [21]:
from demo import make_animation
from skimage import img_as_ubyte

predictions = make_animation(source_image, driving_video, generator, kp_detector, relative=True)

  0%|          | 0/265 [00:00<?, ?it/s]

In [22]:

def display(source, driving, generated=None):
    fig = plt.figure(figsize=(8 + 4 * (generated is not None), 6))

    ims = []
    for i in range(len(driving)):
        cols = [source]
        cols.append(driving[i])
        if generated is not None:
            cols.append(generated[i])
        im = plt.imshow(np.concatenate(cols, axis=1), animated=True)
        plt.axis('off')
        ims.append([im])

    ani = animation.ArtistAnimation(fig, ims, interval=50, repeat_delay=1000)
    plt.close()
    return ani

HTML(display(source_image, driving_video, predictions).to_html5_video())